# Fuzzy switches — filtering, smoothing and estimation

In a switching system the regime $R_n$ usually takes a finite number of values. In
a **fuzzy** switching system it takes its values in $[0, 1]$: the two *hard* regimes
0 and 1 keep a positive probability, and in between the system is a mixture of the
two, with a density (Bouyahia, Derrode, Pieczynski, *IEEE Trans. Fuzzy Systems* 28(8),
2020).

This notebook uses the sub-package `awesometmc.fuzzy` to

1. specify a model and simulate it,
2. **filter** it: $\mathbb{E}[X_n \mid y_{1:n}]$ and the jump $\hat r_n$,
3. **smooth** it: the same given all the observations $y_{1:N}$,
4. compare with the *hardened* two-regime model,
5. **estimate** a model from a sample whose regimes are known.

The key fact: once the integrals over $[0, 1]$ are discretised in $F$ steps, the
fuzzy model is a hard switching model with $K = F + 2$ regimes whose parameters
depend on the **pair** $(r_n, r_{n+1})$. When the observation does not read the
state ($C = 0$), its filter and its smoother are exact and fast.

## Setup

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

from awesometmc.experiments.fuzzy_cgofmsm import paper_gaussian_model
from awesometmc.fuzzy import FMC1, FuzzyCGOMSM, fit_supervised
from awesometmc.utils.exceptions import ParamError


def run_filter(disc, y):
    # Filtered state estimates and regime posteriors of a discretised model.
    filt = disc.filter()
    out = [filt.step(v) for v in y]
    return np.array([o.E_x.ravel() for o in out]).ravel(), np.array([o.pi for o in out])


def mse(a, b):
    return float(np.mean((np.ravel(a) - np.ravel(b)) ** 2))

## 1. A model

A model is a **law of the regime chain** and **Gaussian laws given the regimes**.

- `FMC1(alpha, eta, delta)` is the first law of the paper: mass `alpha` on the pairs
  $(0,0)$ and $(1,1)$, `beta` on $(0,1)$ and $(1,0)$ (deduced), and elsewhere a
  density $\eta + (\delta - \eta)\,|r_1 - r_2|$.
- The Gaussian part is given at the hard regimes only: two means $M(0), M(1)$ of
  $Z = (X, Y)$ and four covariances $\Gamma(i, j)$ of $(Z_1, Z_2)$. In between,
  means are interpolated linearly and covariances bilinearly.

We take the numerical values of Section VI of the paper.

In [ ]:
gauss = paper_gaussian_model()
law = FMC1(alpha=0.07, eta=0.16, delta=0.05)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    model = FuzzyCGOMSM(law, means=(gauss.M0, gauss.M1), gammas=gauss.Gamma)

print(model)
print(f"P(hard regime) = {law.p_hard:.3f}    P(fuzzy regime) = {law.p_fuzzy:.3f}")
print(f"block C at the corners, as printed in the paper: {model.corner_residual():.3f}")
print(f"warning: {str(caught[0].message)[:95]}...")

The warning is expected. With the default `interpolation="cgo"`, the model keeps
$C = 0$ for every pair of regimes, which is what makes the filter exact. Two of the
four matrices printed in the paper do not satisfy that identity (the block $C$
reaches 0.146), so the model corrects them and says so. `interpolation="bilinear"`
keeps the matrices as printed; the filter is then an approximation.

## 2. Simulation

`simulate` draws the regimes in $[0, 1]$, not on a grid.

In [ ]:
N = 300
r, x, y = model.simulate(N=N, seed=2)

fig, axes = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
axes[0].plot(r, color="C3", lw=1)
axes[0].set_ylabel("regime $r_n$")
axes[1].plot(np.ravel(y), color="0.6", lw=1, label="observation $y_n$")
axes[1].plot(np.ravel(x), color="C0", lw=1, label="hidden state $x_n$")
axes[1].set_xlabel("n")
axes[1].legend(loc="upper right", ncol=2)
plt.tight_layout()

## 3. Filtering

`discretise(F)` returns the model on $K = F + 2$ regime values: the two hard ones
and $F$ midpoints. It is an ordinary `PairGSSParams`, filtered by `GSSFilter`.
The jump is estimated by the MPM rule of the paper (`mpm`): first decide between
"hard 0", "hard 1" and "fuzzy", then, if fuzzy, take the posterior mean.

In [ ]:
disc = model.discretise(F=5)
print(f"K = {disc.K} regime values: {np.round(disc.nodes, 2)}")
print(f"filter mode: {disc.filter().mode!r}")

E_filt, pi_filt = run_filter(disc, y)
r_filt = disc.mpm(pi_filt)
print(f"MSE of the state: {mse(E_filt, x):.3f}    MSE of the jumps: {mse(r_filt, r):.3f}")

## 4. Smoothing

`smoother` uses all the observations at once. For this family it is exact and
cheap: the future informs the state only through the regimes, so the smoothed law
is the *filtered* per-regime summaries re-weighted by the smoothed regime
probabilities.

In [ ]:
E_smooth, Var_smooth, gamma, log_lik = disc.smoother(y)
r_smooth = disc.mpm(gamma)
print(f"log-likelihood of the observations: {log_lik:.2f}")
print(f"state  MSE   filter {mse(E_filt, x):.3f}   smoother {mse(E_smooth, x):.3f}")
print(f"jumps  MSE   filter {mse(r_filt, r):.3f}   smoother {mse(r_smooth, r):.3f}")

sl = slice(0, 120)
fig, axes = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
axes[0].plot(r[sl], color="C3", lw=2, alpha=0.5, label="true")
axes[0].plot(r_filt[sl], color="C1", lw=1, label="filtered")
axes[0].plot(r_smooth[sl], color="C2", lw=1, label="smoothed")
axes[0].set_ylabel("regime")
axes[0].legend(loc="upper right", ncol=3)
axes[1].plot(np.ravel(x)[sl], color="C0", lw=2, alpha=0.5, label="true")
axes[1].plot(E_filt[sl], color="C1", lw=1, label="filtered")
axes[1].plot(E_smooth.ravel()[sl], color="C2", lw=1, label="smoothed")
axes[1].set_ylabel("state")
axes[1].set_xlabel("n")
plt.tight_layout()

On one trajectory the two estimates can be close. The averages of the next section
show what smoothing brings: a clear gain on the jumps, a small one on the state,
since the future reaches the state through the regimes only.

## 5. Is the fuzzy model worth it?

`harden()` is the two-regime model a user would write without fuzzy regimes: the
hard Gaussian laws, and the chain obtained by rounding each regime to 0 or 1. We
average over a few trajectories, and look at the effect of $F$.

In [ ]:
runs = [model.simulate(N=400, seed=100 + k) for k in range(20)]


def study(disc):
    # Mean errors over the runs: state and jumps, filtered then smoothed.
    errors = []
    for r_k, x_k, y_k in runs:
        E_k, pi_k = run_filter(disc, y_k)
        Es_k, _, gamma_k, _ = disc.smoother(y_k)
        errors.append(
            [mse(E_k, x_k), mse(Es_k, x_k), mse(disc.mpm(pi_k), r_k), mse(disc.mpm(gamma_k), r_k)]
        )
    return np.mean(errors, axis=0)


print("                    state MSE           jump MSE")
print("model          K    filter  smoother    filter  smoother")
print("hardened       2    {:.3f}   {:.3f}       {:.3f}   {:.3f}".format(*study(model.harden())))
for F in (0, 1, 3, 5, 9):
    row = study(model.discretise(F=F))
    print(f"fuzzy, F = {F}  {F + 2:2d}    " + "{:.3f}   {:.3f}       {:.3f}   {:.3f}".format(*row))

With $F = 0$ the fuzzy model has the same two regimes as the hardened one, but
keeps the right chain. A few fuzzy steps bring most of the gain on the jumps; the
errors then level off, as reported in the paper. The cost of one step grows like
$K^2$.

## 6. Estimating a model from labelled data

When the regimes of a sample are known, `fit_supervised(r, x, y)` estimates the
whole model:

- the law of the jumps, from the pairs $(r_n, r_{n+1})$;
- the means and the four corner covariances, by least squares on **all** the
  samples, fuzzy ones included (`method="regression"`).

In [ ]:
r_tr, x_tr, y_tr = model.simulate(N=20_000, seed=7)
fit = fit_supervised(r_tr, x_tr, y_tr, law="fmc1")

print(
    f"law     true  alpha={law.alpha:.3f} beta={law.beta:.3f} eta={law.eta:.3f} delta={law.delta:.3f}"
)
est = fit.law
print(
    f"        est.  alpha={est.alpha:.3f} beta={est.beta:.3f} eta={est.eta:.3f} delta={est.delta:.3f}"
)
print(f"means   true  M(0)={model.mean(0.0)}  M(1)={model.mean(1.0)}")
print(
    f"        est.  M(0)={np.round(fit.gaussian.means[0], 2)}  M(1)={np.round(fit.gaussian.means[1], 2)}"
)
worst = max(
    np.abs(fit.gaussian.gammas[c] - model.gamma(*map(float, c))).max() for c in fit.gaussian.gammas
)
print(f"largest error on the four corner covariances: {worst:.3f}")

E_true, _ = run_filter(model.discretise(F=5), y)
E_fit, _ = run_filter(fit.model.discretise(F=5), y)
print(
    f"state MSE on the trajectory of section 2: true model {mse(E_true, x):.3f}, fitted model {mse(E_fit, x):.3f}"
)

### A pseudo ground truth, as in Section VII of the paper

On real data the regimes are not observed. The paper lays a **pseudo ground truth**
over the series: hard 0 during the night, hard 1 in the afternoon, and a linear ramp
in between. We reproduce that protocol on synthetic data (a "temperature" to
estimate from a "power").

With such regimes the hard pairs $(0, 1)$ and $(1, 0)$ never occur, since the regime
always goes through fuzzy values. The purely empirical estimators over hard samples
(`method="hard"`) therefore cannot give the four corners; the regression can.

In [ ]:
hours = np.arange(40 * 24) % 24
r_day = np.interp(hours, [0, 1, 5, 13, 17, 24], [0.25, 0.0, 0.0, 1.0, 1.0, 0.25])
rng = np.random.default_rng(3)
temp = 60 + 25 * r_day + rng.normal(0, 2, r_day.size)
power = 100 + 80 * r_day + 1.5 * (temp - 60 - 25 * r_day) + rng.normal(0, 4, r_day.size)

try:
    fit_supervised(r_day, temp, power, method="hard")
except ParamError as err:
    print("method='hard' :", err)

fit_day = fit_supervised(r_day, temp, power)
print("hard pairs seen:", fit_day.gaussian.n_hard_pairs)
print("rank of the design of the four corners:", fit_day.gaussian.cross_rank, "of 4")

disc_day = fit_day.model.discretise(F=5)
E_f, _ = run_filter(disc_day, power)
E_s, _, gamma_day, _ = disc_day.smoother(power)
print(
    f"temperature MSE: filter {mse(E_f, temp):.2f}, smoother {mse(E_s, temp):.2f} (variance {np.var(temp):.1f})"
)

sl = slice(0, 96)
fig, axes = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
axes[0].plot(r_day[sl], color="C3", lw=2, alpha=0.5, label="pseudo ground truth")
axes[0].plot(disc_day.mpm(gamma_day)[sl], color="C2", lw=1, label="smoothed jump")
axes[0].set_ylabel("regime")
axes[0].legend(loc="upper right")
axes[1].plot(temp[sl], color="C0", lw=2, alpha=0.5, label="true")
axes[1].plot(E_s.ravel()[sl], color="C2", lw=1, label="smoothed")
axes[1].set_ylabel("state")
axes[1].set_xlabel("hour")
axes[1].legend(loc="upper right")
plt.tight_layout()

`fit.gaussian` holds the diagnostics of the estimate: the hard samples and pairs
seen, the rank of the design of the four corners (below 4 when the sample does not
separate them; the least-norm solution is then returned), and whether a corner had
to be shrunk to stay a valid covariance (`cross_shrink`).

## Going further

- `FMC2` is the second law of the paper (a band $|r_1 - r_2| \le \delta$);
  `FuzzyGrid.custom` sets your own quadrature nodes and weights.
- The discretised model is a `PairGSSParams`: everything in `awesometmc.filter`
  that accepts one applies (`GSSFilter` modes `"cgo"` and `"gpb2"`,
  `regime_smoother(..., kernel="c0")`, `chain_log_likelihood`).
- `python -m awesometmc.experiments.fuzzy_cgofmsm_paper` reproduces Fig. 5 and
  Table I of the paper.